# Bronze Layer — Source Blob → Charging Sessions

Copies raw CSV files from the instructor-provided source blob storage into the Bronze Volume on Unity Catalog, preserving the exact source directory structure.

**Source path layout:**
```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── realtime/
        └── charging_sessions/
              └── YYYY/MM/DD/HH/
                    └── sessions_YYYYMMDD_HHMM.csv
```

**Bronze target path layout (mirrors source exactly):**
```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── realtime/
        └── charging_sessions/
              └── YYYY/MM/DD/HH/
                    └── sessions_YYYYMMDD_HHMM.csv
```

**Load modes — controlled by one variable `LOAD_MODE` in Cell 2:**
| `LOAD_MODE` | What it copies |
|---|---|
| `full` | All files under `realtime/charging_sessions/` |
| `incremental` | Only files under `realtime/charging_sessions/YYYY/MM/DD/HH/` matching `LOAD_YEAR`, `LOAD_MONTH`, `LOAD_DAY`, `LOAD_HOUR` |

## Cell 1 — Mount source blob storage

Configures Spark to authenticate to `dataenggdailystorage` using a SAS token stored in the Databricks secret scope. This must be run every session — Spark config does not persist across cluster restarts.

- `dbutils.secrets.get` reads the SAS token from Key Vault via the `kv-ev-scope` secret scope
- `spark.conf.set` registers the SAS credential for the `wasbs://` driver so all subsequent reads use it automatically
- `SOURCE_ROOT` is the base `wasbs://` URL — all file paths are built by appending to this

In [ ]:
SCOPE = "kv-ev-scope"
SOURCE_ACCOUNT = "dataenggdailystorage"
SOURCE_CONTAINER = "source"

sas_token = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{SOURCE_CONTAINER}.{SOURCE_ACCOUNT}.blob.core.windows.net",
    sas_token
)

SOURCE_ROOT = f"wasbs://{SOURCE_CONTAINER}@{SOURCE_ACCOUNT}.blob.core.windows.net"

print(f"Source root : {SOURCE_ROOT}")
print("SAS token   : [REDACTED]")
print("Source blob authenticated — OK")

## Cell 2 — Set load mode and path variables

**This is the only cell you change between full and incremental runs.**

- `LOAD_MODE = "full"` → copies everything under `realtime/charging_sessions/`
- `LOAD_MODE = "incremental"` → copies only the folder matching the year/month/day/hour you set below

For incremental, set `LOAD_YEAR`, `LOAD_MONTH`, `LOAD_DAY`, `LOAD_HOUR` to the partition you want.
These values match the folder names in the source blob exactly (zero-padded: `06` not `6`).

The Bronze Volume path is the Unity Catalog external volume mounted at:
`/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/`

In [ ]:
LOAD_MODE = "incremental"   # "full" or "incremental"

LOAD_YEAR  = "2026"
LOAD_MONTH = "06"
LOAD_DAY   = "01"
LOAD_HOUR  = "06"

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
BASE_SUBPATH  = "realtime/charging_sessions"

if LOAD_MODE == "full":
    source_path = f"{SOURCE_ROOT}/{BASE_SUBPATH}/"
    bronze_path = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/"
else:
    partition   = f"{LOAD_YEAR}/{LOAD_MONTH}/{LOAD_DAY}/{LOAD_HOUR}"
    source_path = f"{SOURCE_ROOT}/{BASE_SUBPATH}/{partition}/"
    bronze_path = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{partition}/"

print(f"LOAD_MODE   : {LOAD_MODE}")
print(f"Source path : {source_path}")
print(f"Bronze path : {bronze_path}")

## Cell 3 — List source files to be copied

Lists all files at the source path before copying. Use this to confirm the path is correct and the expected files are visible.

- `dbutils.fs.ls` returns a list of `FileInfo` objects with `.name`, `.path`, `.size`
- For `full` mode this will recurse into all year/month/day/hour subdirectories
- For `incremental` mode this shows only the files in the single hour folder

In [ ]:
def list_files_recursive(path):
    items = dbutils.fs.ls(path)
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

source_files = list_files_recursive(source_path)

print(f"Files found at source ({LOAD_MODE} mode): {len(source_files)}")
for f in source_files:
    size_kb = round(f.size / 1024, 1)
    print(f"  {f.path}  [{size_kb} KB]")

## Cell 4 — Copy files to Bronze Volume

Copies each file from the source blob to the Bronze Volume, recreating the full directory structure under the Bronze base path.

- `dbutils.fs.cp` copies a single file. The third argument `True` enables recursive copy for directories, but here we copy file-by-file so we can log each result individually
- The relative path is extracted by stripping the `source_path` prefix, then appended to `bronze_path` — this is what preserves the folder structure
- Files that already exist at the destination are overwritten (no-op if identical — blob storage has no version conflict)

In [ ]:
copied  = []
skipped = []

for file_info in source_files:
    relative_path = file_info.path.replace(source_path, "")
    dest_path     = bronze_path + relative_path

    try:
        dbutils.fs.cp(file_info.path, dest_path)
        copied.append(dest_path)
        print(f"  COPIED  {relative_path}")
    except Exception as e:
        skipped.append((file_info.path, str(e)))
        print(f"  FAILED  {relative_path} — {e}")

print(f"\nCopy complete: {len(copied)} copied, {len(skipped)} failed")

## Cell 5 — Verify files in Bronze Volume

Lists the files now present in the Bronze Volume at the destination path. Compare this against Cell 3 output — every file in the source should appear here.

- For `full` mode: all year/month/day/hour subfolders under `realtime/charging_sessions/` should be present
- For `incremental` mode: only the single hour folder and its CSV should be present

In [ ]:
bronze_files = list_files_recursive(bronze_path)

print(f"Files in Bronze Volume ({LOAD_MODE} mode): {len(bronze_files)}")
for f in bronze_files:
    size_kb = round(f.size / 1024, 1)
    print(f"  {f.path}  [{size_kb} KB]")

assert len(bronze_files) == len(source_files), (
    f"Count mismatch: source={len(source_files)}, bronze={len(bronze_files)}"
)

## Cell 6 — Read one CSV file and inspect schema

Reads the first copied CSV from the Bronze Volume into a Spark DataFrame to confirm the file is readable and the columns are as expected.

- `header=True` uses the first CSV row as column names
- `inferSchema=True` lets Spark detect column types (string, int, timestamp, etc.) — useful at this stage; Silver layer will enforce explicit types
- `printSchema()` prints the column tree with inferred types
- `.limit(5)` reads only 5 rows — avoids scanning large files just to verify the read works

In [ ]:
if not bronze_files:
    print("No files found in Bronze Volume — re-run Cell 4.")
else:
    sample_file = bronze_files[0].path
    print(f"Reading: {sample_file}")

    df = spark.read.option("header", True).option("inferSchema", True).csv(sample_file)

    print(f"Row count : {df.count():,}")
    print(f"Columns   : {len(df.columns)}")
    df.printSchema()
    display(df.limit(5))

## Cell 7 — Summary

Prints a final run summary: load mode, source path, Bronze Volume path, file count, and any failures. Save this output as a screenshot or copy it to your run log.

This notebook does **not** write a Delta table — it mirrors raw CSV files into the Bronze Volume exactly as received. The Silver layer notebook will read these CSVs, apply schema, deduplicate, and write Delta.

In [ ]:
print("=" * 60)
print("BRONZE BLOB MIGRATION — RUN SUMMARY")
print("=" * 60)
print(f"Load mode      : {LOAD_MODE}")
print(f"Source path    : {source_path}")
print(f"Bronze path    : {bronze_path}")
print(f"Files copied   : {len(copied)}")
print(f"Files failed   : {len(skipped)}")
if skipped:
    print("\nFailed files:")
    for src, err in skipped:
        print(f"  {src}  →  {err}")
print("=" * 60)
print("Next step: Silver layer reads from Bronze Volume and writes Delta")